In [79]:
import numpy as np
import qiskit.quantum_info as qi
from processtensor import QTensor, QuantumState, UnitarySE, ProcessTensor
from qiskit.visualization import plot_state_city, plot_state_hinton

In [80]:
def pauli2_pt_choi(pt_choi):
    """"""
    n_qubits = int(np.log2(pt_choi.shape[0]) / 2)  # Number of system qubits
    twirled = np.zeros_like(pt_choi, dtype=np.complex128)
    factor = 1 / (4 ** n_qubits)  # Normalization factor for
    
    basis_strings = qi.pauli_basis(n_qubits)
    basis_superop_products = [superop_from_pauli_string(basis_string) for basis_string in basis_strings]
    
    for superop_product in basis_superop_products:
        
        twirled += factor * superop_product @ pt_choi @ superop_product.conj().T
        
    return twirled

def superop_from_pauli_string(pauli_string):
    n_qubits = len(pauli_string)
    
    superops = []
    
    for i in range(n_qubits):
        matrix = pauli_string[i].to_matrix()
        superop = np.kron(matrix, np.conj(matrix.T))
        superops.append(superop)
        
    return multi_kron(superops)

def multi_kron(Ms):
    """Compute the tensor product of a list of matrices."""
    prod = Ms[0]
    for i in range(1, len(Ms)):
        prod = np.kron(prod, Ms[i])
    return prod

In [81]:
dS = 5
dE = 25
d = dS * dE

E_init = np.eye(dE)

rho = qi.random_density_matrix(dE).data
U1 = qi.random_unitary(d).data
U2 = qi.random_unitary(d).data
# U1 = np.eye(d)
# U2 = np.eye(d)

q0 = QTensor(QuantumState.from_matrix(rho, dS=1))
q1 = QTensor(UnitarySE.from_unitary(U1, dS, dE))
q2 = UnitarySE.from_unitary(U2, dS, dE)

pt = q0 @ q1 @ q2
pt = ProcessTensor(pt.trace_subsystem(pt.ndim - 1))
choi_matrix = pt.choi_matrix
# choi_matrix = qi.random_density_matrix(dS**4).data

print(f"Process tensor is completely positive: {pt.is_cp()}")
print(f"QTensor dims: {pt.dims}")
print(f"Choi matrix dims: {choi_matrix.shape}")

# Apply MPO-style twirling that preserves bond dimension
# twirled = pauli2_pt_choi(choi_matrix)
# print(f"Twirled process tensor dims: {twirled.shape}")

Process tensor is completely positive: True
QTensor dims: (1, 25, 25, 25, 25, 1)
Choi matrix dims: (625, 625)


In [82]:
#plot_state_hinton(choi_matrix, title="Original Choi matrix")

In [83]:
#plot_state_hinton(twirled, title="MPO-Twirled Choi matrix")

In [84]:
def decompose_choi(choi_matrix):

    d = int(np.sqrt(choi_matrix.shape[0]))
    transposed = choi_matrix.reshape(d, d, d, d).transpose(0, 2, 1, 3).reshape(d**2, d**2)
    U, S, Vh = np.linalg.svd(transposed)
    n = np.sum(S > 1e-10)
    U = U[:, :n]
    S = S[:n]
    S_sqrt = np.sqrt(S)
    Vh = Vh[:n, :]
    
    U = U @ np.diag(S_sqrt)
    V = np.diag(S_sqrt) @ Vh

    return U, S, V

In [85]:
# Compare bond dimensions (rank of Choi matrices)
U1, S1, V1 = decompose_choi(choi_matrix)
n1 = np.sum(S1 > 1e-8)

U2, S2, V2 = decompose_choi(twirled)
n2 = np.sum(S2 > 1e-8)

print(f"=== Bond Dimension Comparison ===")
print(f"Original Choi matrix rank (bond dimension): {n1}")
print(f"MPO-twirled Choi matrix rank (bond dimension): {n2}")
print(f"Bond dimension preserved: {n1 == n2}")

print(f"\nOriginal singular values: {S1[S1 > 1e-10]}")
print(f"Twirled singular values: {S2[S2 > 1e-10]}")

=== Bond Dimension Comparison ===
Original Choi matrix rank (bond dimension): 601
MPO-twirled Choi matrix rank (bond dimension): 13
Bond dimension preserved: False

Original singular values: [1.05526315e+00 1.50314002e-01 1.45200141e-01 1.43928636e-01
 1.42783482e-01 1.40101294e-01 1.38120184e-01 1.37206675e-01
 1.36313842e-01 1.34840255e-01 1.33856192e-01 1.32296224e-01
 1.31344785e-01 1.30573193e-01 1.28982214e-01 1.28052969e-01
 1.27152373e-01 1.26501627e-01 1.25906734e-01 1.24761559e-01
 1.24443368e-01 1.23739639e-01 1.21884841e-01 1.21289230e-01
 1.19413861e-01 1.19295083e-01 1.18755258e-01 1.18302592e-01
 1.17977659e-01 1.16840176e-01 1.16290057e-01 1.15411005e-01
 1.15118150e-01 1.14399843e-01 1.13875100e-01 1.13549353e-01
 1.13025619e-01 1.11493593e-01 1.11185700e-01 1.10597085e-01
 1.09847038e-01 1.09632639e-01 1.09096447e-01 1.08456878e-01
 1.07632990e-01 1.07247458e-01 1.06957219e-01 1.06883031e-01
 1.06501154e-01 1.05099430e-01 1.04429823e-01 1.04053382e-01
 1.03453191e-01 